[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritterlab/health_innovation_lab/blob/main/tutorial_2_2_images_and_cnn.ipynb)

# Health Innovation Lab (HIL) — Tutorial 2.2 (optional)
## Images, MRI and convolutional neural networks

Tutors: *(add your names/emails here)*

*Adapted for HIL from two Charité "AI in Medicine" tutorials (July 2024): "Python Programming: computer vision, MRI and deep learning" by Marc-André Schulz (AG Ritter) and "Python Programming: Deep Learning using Keras" by Moritz Seiler (AG Machine Learning in Clinical Neuroimaging).*

This is an **optional extra** for everyone who wants to go further than Tutorial 2. In Tutorial 2 you worked with *tables* of numbers. Here we work with **images**: first you see how a computer "sees" a picture and a brain scan, and run a trained neural network on a real MRI. Then you **build and train your own** neural network.

<font color=green>This notebook is exercise-heavy on purpose. You are not expected to finish everything — work at your own pace, try first, and ask questions whenever you are stuck.</font>

***

## 1. Aims of this session

- How to work with images and MRI data in a programming language.
- How to use a trained convolutional neural network (CNN) on an MRI scan.
- How to build, train and evaluate your own CNN.

## 2. Learning goals

### Theory
- Images (and MRI scans) are arrays of numbers
- Convolutional filters (kernels)
- Convolutional neural networks (CNNs): layers, parameters, training
- Training and validation curves, over-fitting

### Practical
- Open, inspect, slice and modify images with *numpy* and *matplotlib*
- Apply and design convolution filters by hand
- Load an MRI scan with *nibabel* and look at it from three directions
- Preprocess an MRI (mask, resize) and run a pretrained 3D CNN on it
- Build, compile, train and evaluate your own CNN with *Keras*
- Look at what your model gets wrong

## 3. References
- Deep learning
  - https://www.deeplearningbook.org/
  - https://cs231n.github.io/
- Keras
  - https://keras.io/guides/
  - https://www.tensorflow.org/tutorials/keras/classification
- nibabel (reading MRI files): https://nipy.org/nibabel/

***

## 4. How this notebook works

Code cells that contain `# TODO` need your code. Below each exercise there is a collapsible **Solution** box — only open it if you are really stuck, and ask questions first! The exercises build on each other, so if you skip one, copy its solution before you continue. This whole notebook is optional — do as much of it as you like.

*Two libraries, one idea:* Part A uses *PyTorch* (the pretrained network we use was written in it), Part B uses *Keras*/*TensorFlow* (it is quick to read and write). You do not need to know either in depth — layers, filters, loss functions and training work the same way in both.

### Setup

Google Colab already includes numpy, scipy, matplotlib, scikit-learn, PyTorch and TensorFlow. Only *nibabel* (for reading MRI files) may be missing, so we install it — this does nothing if it is already there. *(Running locally? You additionally need `pip install torch tensorflow`.)*

In [ ]:
!pip install -q nibabel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from PIL import Image

***
## 5. Part A — How computers see images, and a CNN on a real brain scan

### 5.1 Opening an image with Python

We download a photo and open it. There are many tools to read a jpeg image into Python; here we use the common *Pillow* (PIL) package.

In [ ]:
!mkdir -p downloads
!wget -q -nc -P downloads https://raw.githubusercontent.com/ritterlab/ai_in_medicine/master/data/car.jpg
image_path = "downloads/car.jpg"

image = Image.open(image_path)

> **Exercise A1**: Show the image using matplotlib.

In [ ]:
# TODO: show the image (hint: plt.imshow)


### 5.2 Images on computers

Computers store everything in binary code, only 1s and 0s. On a higher level, an operating system stores data in a more structured fashion — which lets us understand and modify it much better. How are images stored?

The result of Exercise A1 shows the image in a 2D grid, i.e. it has an X and a Y axis. A 2D grid can be seen as a simple matrix. In Python we often store any kind of matrix, such as a vector, a matrix or a tensor, as an *array*.

> **Exercise A2**: Use numpy to convert the image into an array called `image_array`.

In [ ]:
image_array = None  # TODO (hint: np.array)


> **Exercise A3**: Print the shape of the array, its data type (`.dtype`), and its smallest and largest value.

In [ ]:
# TODO


You will notice that the array is in fact 3D, not 2D! The last dimension represents the colour: it has 3 *channels* — red, green and blue. And every value lies between 0 and 255 — an 8-bit number per pixel and channel.

If the image is simply an array, we should be able to have a look at what is inside! Printing everything would be far too long, so let's look at the first 5×5 pixels using *slicing*.

> **Exercise A4**: Print the first 5×5 pixels of `image_array`.
>
> Hint: use brackets `[]` for selecting, `,` to separate the dimensions and `:X` to select from the beginning up to X.

In [ ]:
# TODO


You should see 5 rows, nested within 5 larger rows. Each row contains three values: red, green and blue!

### 5.3 Modifying image contents

Next, let's see how modifying the array changes the picture.

> **Exercise A5**: Set the first 100×100 pixels to zero and plot the image. What colour do you get?

In [ ]:
zeroed_image = np.copy(image_array)  # copy the values to a new variable
# TODO: select the top-left 100x100 pixels and set them to 0

plt.imshow(zeroed_image)
plt.show()

If we set all three RGB channels to zero, the pixels turn black.

> **Exercise A6**: Remove the red colour from the whole image (set the red channel to 0).
>
> Hint: use `:` on the two spatial dimensions and `0` on the channel dimension.

In [ ]:
no_red_image = np.copy(image_array)
# TODO

plt.imshow(no_red_image)
plt.show()

> **Exercise A7**: Now show *only* the red channel (set the other two channels to 0). Can you do it in one line?

In [ ]:
red_only_image = np.copy(image_array)
# TODO

plt.imshow(red_only_image)
plt.show()

### 5.4 Filters: what a convolutional layer does

Next, we apply a simple **filter/kernel** to the image, which is typical of what a convolutional neural network does. The filter (a small matrix, much smaller than the image) is moved across all spatial positions of the image — this is often called a *sliding window*. At every position we multiply the filter with the pixels underneath it and add everything up; that sum becomes one pixel of the new image.

First we create two different kinds of kernel.

In [ ]:
from scipy.ndimage import correlate

horizontal_kernel = np.array([
    [ 1,  1,  1],
    [ 0,  0,  0],
    [-1, -1, -1]
])

vertical_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

The two kernels above are typical *edge detector* filters; edge detection is important to find the boundaries between different objects. By just looking at the numbers and considering how the kernels slide over the image, one can understand that the horizontal kernel highlights places where there is a sharp difference in values *above and below* the line, while the vertical kernel does the same from left to right.

To keep things simple we use only one colour channel (red) as a grey-scale picture. We convert it to decimal numbers (`float`) so that negative filter results are not cut off.

*(A technical note: strictly speaking, CNN layers compute a "cross-correlation", i.e. a convolution without flipping the kernel. In `scipy` this is the function `correlate`. The idea is the same.)*

In [ ]:
gray = image_array[:, :, 0].astype(float)

> **Exercise A8**: Apply both kernels to `gray`. Hint: `correlate(image, kernel)`.

In [ ]:
horizontally_filtered = None  # TODO
vertically_filtered = None    # TODO


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(gray, cmap='gray');                  axes[0].set_title('original (red channel)')
axes[1].imshow(horizontally_filtered, cmap='gray'); axes[1].set_title('horizontal kernel')
axes[2].imshow(vertically_filtered, cmap='gray');   axes[2].set_title('vertical kernel')
for ax in axes:
    ax.axis('off')
plt.show()

As you can see, the filters highlight the horizontal and vertical edges in the image. However, since the image contains many "grainy" areas like the road and the buildings, as well as diagonal lines such as the road markings, it is quite difficult to make sense of the image content from these results. Nevertheless, the car, as the main object, is highlighted nicely in both versions.

Let's make sure we really understand what happens at *one* position. The filter response at pixel (100, 100) is the sum of (the 3×3 pixels around it × the kernel).

> **Exercise A9**: Compute the response of the horizontal kernel at pixel (100, 100) by hand: take the 3×3 patch `gray[99:102, 99:102]`, multiply it element-wise with `horizontal_kernel` and sum it up. Compare to the value in `horizontally_filtered`.

In [ ]:
patch = gray[99:102, 99:102]
manual = None  # TODO

print('by hand: ', manual)
print('scipy:   ', horizontally_filtered[100, 100])

> **Exercise A10**: Design your own kernel. A *blur* filter replaces every pixel by the average of its neighbourhood: a 3×3 kernel where every entry is 1/9. Create it (hint: `np.ones((3, 3)) / 9`), apply it to `gray` and plot the result. Then try a bigger one (e.g. 15×15, entries 1/225). What happens?

In [ ]:
# TODO


> **Interpretation exercise**
>
> We wrote the numbers of these filters by hand. For recognising something abstract — a car, or a brain disease — nobody can write down the right filter numbers. Where could they come from instead?

*Your answer:*

> ...


### 5.5 MRI scans are arrays as well

A photo is a 2D grid with 3 colour channels. An MRI scan is a **3D grid** with one value per *voxel* (3D pixel) — for example how bright the tissue appears at that position. In Tutorial 2 you used volumes such as `Hippocampus` and `Ventricles`; numbers like these are derived from scans like the one we load now.

We download one MRI scan (about 47 MB — this takes a few seconds) together with a brain mask that we need later.

In [ ]:
!wget -q -nc -P downloads https://raw.githubusercontent.com/ritterlab/ai_in_medicine/master/data/MRI_subject.nii.gz
!wget -q -nc -P downloads https://raw.githubusercontent.com/ritterlab/ai_in_medicine/master/data/binary_brain_mask.nii.gz

> **Exercise A11**: Load the MR image with `nib.load(...)` into a variable called `nifti`. The file is `downloads/MRI_subject.nii.gz`.

In [ ]:
nifti = None  # TODO


A NIfTI file does not only contain the numbers but also meta information, e.g. how big one voxel is (in millimetres). To access the numbers themselves we use the `get_fdata()` function.

In [ ]:
print("Voxel size in mm:", nifti.header.get_zooms())

mri_data = nifti.get_fdata()

> **Exercise A12**: Print the shape of the MRI, the total number of values (`.size`), and its smallest and largest value.

In [ ]:
# TODO


> **Interpretation exercise**
>
> In Tutorial 2, one patient was described by about 5 numbers (e.g. `MMSE`, `Hippocampus`, ...). How many numbers does this one MRI contain? What could that mean for a model that learns from raw scans instead of from a small table?

*Your answer:*

> ...


> **Exercise A13**: Plot a single slice of the 3D array with `plt.imshow`. To plot a 3D MRI as a 2D image we need to select a single slice: use the all-selector `:` for two dimensions and a number for the third. A slice in the middle makes sense, as the slices at the very edge are typically empty. Use `cmap='gray'` to paint the image in grey-scale.

In [ ]:
# TODO


> **Exercise A14**: Show the three standard views of the brain next to each other (one subplot each), by slicing through the middle of each of the three axes, and give each plot a title: *sagittal* (side view), *coronal* (front view) or *axial* (view from above). Which axis gives which view?
>
> Hints: `plt.subplots(1, 3)`; `mri_data.shape[0] // 2` is the middle of the first axis. `imshow` draws the first axis downwards, so a slice may look rotated — transposing it (`.T`) and adding `origin='lower'` turns it upright.

In [ ]:
# TODO


### 5.6 A convolutional neural network on an MRI scan

You have just seen that a filter slides over an image and highlights certain patterns. **Convolutional neural networks** apply such filters in a hierarchical, layer-wise fashion. Using reasoning to come up with simple edge detector filters like above, or even face recognition filters (see the [Viola-Jones algorithm](https://de.wikipedia.org/wiki/Viola-Jones-Methode)), seems plausible, but doing that for abstract concepts such as cars, or brain diseases, is impossible. CNNs are able to **learn** those filters automatically using backpropagation. Interestingly, it has been shown that CNNs trained on images often learn exactly this kind of edge detection filter in their earliest layers. Since different filters are combined along the depth of the network, more challenging concepts can be extracted by using deep networks.

Next, we use a trained neural network to predict on the MR image from above whether the subject has Alzheimer's disease or not.

#### Preprocessing

The network was trained on brain images without skull and with a fixed size. So first we have to bring our scan into the same form.

> **Exercise A15**: Load the brain mask `downloads/binary_brain_mask.nii.gz` into a variable `mask`. Our brain mask is **binary**, meaning it only contains 1s and 0s: a 1 everywhere there is grey or white matter, and a 0 otherwise.

In [ ]:
mask = None  # TODO (you need nib.load(...) and get_fdata() again)


In [ ]:
print(mask.shape)
plt.imshow(mask[:, :, 90], cmap='gray')
plt.show()

> **Exercise A16**: Apply the mask to the MRI (the image still contains the skull!). Think about what multiplying by 0 and by 1 does. Plot a slice of the result.

In [ ]:
masked_mri = None  # TODO

plt.imshow(masked_mri[:, :, 90], cmap='gray')
plt.show()

The network expects inputs of size (96, 114, 96), which is exactly **half** of the size of our MRI. We can use the `zoom` function from `scipy` to match the size.

> **Exercise A17**: Shrink `masked_mri` to half its size with `zoom(input, zoom=0.5)` and check the new shape.

In [ ]:
from scipy.ndimage import zoom

mri_data_zoomed = None  # TODO

print(mri_data_zoomed.shape)

*(Note: the masking together with the zooming is not perfect. Better methods to remove the skull exist, such as FSL BET.)*

#### The network

Below, the class `NeuralNetwork` specifies our CNN, defined in the *PyTorch* framework. You do not need to understand all of it, but you can identify the different building blocks: convolutional, max-pooling and linear (fully connected) layers, as well as dropout, ELU and sigmoid activations.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Convert the zoomed and masked MRI to a PyTorch tensor.
# The network expects 5 dimensions (batch, channel, x, y, z), so we add two
# dimensions of length 1 at the beginning (unsqueeze). We also convert the values to float.
mri_tensor = torch.from_numpy(mri_data_zoomed).unsqueeze(0).unsqueeze(0).float()
print(mri_tensor.shape)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, drp_rate=0.3):
        super(NeuralNetwork, self).__init__()
        self.drp_rate = drp_rate
        self.dropout = nn.Dropout3d(p=self.drp_rate)
        self.Conv_1 = nn.Conv3d(1, 64, kernel_size=3, stride=1, padding=0)
        self.pool_1 = nn.MaxPool3d(kernel_size=3, stride=3, padding=0)
        self.Conv_2 = nn.Conv3d(64, 64, kernel_size=3, stride=1, padding=0)
        self.pool_2 = nn.MaxPool3d(kernel_size=3, stride=2, padding=0)
        self.Conv_3 = nn.Conv3d(64, 64, kernel_size=3, stride=1, padding=0)
        self.Conv_4 = nn.Conv3d(64, 64, kernel_size=3, stride=1, padding=0)
        self.Conv_5 = nn.Conv3d(64, 36, kernel_size=3, stride=1, padding=0)
        self.pool_4 = nn.MaxPool3d(kernel_size=4, stride=2, padding=0)

        self.classifier_scratch = nn.Sequential(
            nn.Linear(1296, 80),
            nn.Linear(80, 1)
        )

    def encode(self, x):
        x = F.elu(self.Conv_1(x))
        h = self.dropout(self.pool_1(x))
        x = F.elu(self.Conv_2(h))
        h = self.dropout(self.pool_2(x))
        x = F.elu(self.Conv_3(h))
        x = F.elu(self.Conv_4(x))
        x = F.elu(self.Conv_5(x))
        h = self.dropout(self.pool_4(x))
        return h

    def forward(self, x):
        x = self.encode(x)
        x = self.flatten(x)
        x = self.classifier_scratch(x)
        x = torch.sigmoid(x)
        return x

    def flatten(self, x):
        return x.view(x.size(0), -1)

We create an instance of the network and load the **pre-trained weights** — the numbers the network learned during training.

In [ ]:
net = NeuralNetwork()

!wget -q -nc -P downloads https://raw.githubusercontent.com/ritterlab/ai_in_medicine/master/data/cnn_params.h5

try:
    state = torch.load("downloads/cnn_params.h5", map_location='cpu')
except Exception:
    # Recent PyTorch versions are strict about old checkpoint files. We trust this file
    # (it comes from the course repository), so we allow the permissive loader.
    # Never do this with files from sources you do not trust!
    state = torch.load("downloads/cnn_params.h5", map_location='cpu', weights_only=False)

print(net.load_state_dict(state, strict=False))   # should print: <All keys matched successfully>

# 'evaluation' mode turns dropout off, which we only want during training
net.eval()

> **Exercise A18**: How many *parameters* (learnable numbers) does this network have? Hint: `net.parameters()` gives all parameter tensors, and `.numel()` counts the numbers in one tensor. Compare with Tutorial 2: the linear regression there had just two parameters (a slope and an intercept).

In [ ]:
n_params = None  # TODO
print(n_params)

The network was trained to tell **Alzheimer's disease from healthy controls**, so it is a *binary classification*. That is why the *sigmoid* activation is used as the output: it squeezes any number into the range between 0 and 1, so the output can be read as a probability-like score. The network was trained with the class encoding:

- 0 — healthy control
- 1 — Alzheimer's disease patient

So any output above 0.5 is classified as AD, and any output below 0.5 as healthy control.

> **Exercise A19**: Use the network to predict on our MR image. Hint: call the network like a function on `mri_tensor`, inside `with torch.no_grad():` (this tells PyTorch that we are not training). Print the output.

In [ ]:
with torch.no_grad():
    output = None  # TODO

print(output.item())

The image should be classified as Alzheimer's disease — with an output of 1.0: the network is "completely sure".

> **Interpretation exercise**
>
> The network says "Alzheimer's disease", with a score of 1.0. Is this proof that the subject has Alzheimer's disease — and is it proof that the network works? Should we believe a network that is 100% sure about a single scan? What would you want to know *before* you let such a network help with real patients? (Think about: the data it was trained on, how it was tested, who the patients were, what happens when it is wrong.)

*Your answer:*

> ...


> **Exercise A20 — Does the network know when it is *not* looking at a brain?** Feed the network two things that are certainly no brains: **(a)** a volume that contains only zeros, **(b)** a volume of random noise with values between 0 and 2000 (the brightness range of our scan). Both need the shape (1, 1, 96, 114, 96). Hint: `torch.zeros(...)` and `torch.rand(...) * 2000`. What does the network answer?

In [ ]:
# TODO: create the two tensors and predict on both (use `with torch.no_grad():` again)


> **Interpretation exercise**
>
> What does the result of A20 tell you about the "1.0" we got for the real scan in A19? Can a network like this say "this is not a brain" or "I don't know"? Where could such a network end up making a wrong decision in a hospital — for example with a bad scan, a different scanner or a different patient population?

*Your answer:*

> ...


***
## 6. Part B — Build and train your own CNN

In Part A somebody else had already trained the network. Where do those weights come from? From **training**: the network starts with random filters and adjusts them again and again so that its predictions get closer to the known labels. Now you do this yourself.

Training on 3D MRI scans takes a lot of data and computing power, so we use a smaller, classic problem: recognising **handwritten digits**. The images are tiny (8×8 pixels, grey-scale) — but they are still arrays of numbers, exactly like the photo and the MRI slices above.

The workflow in *Keras* consists of these steps:
1. **Prepare data**
2. **Build model**
3. **Compile model**
4. **Fit (train) model**
5. **Evaluate model**
6. **Predict**

### 6.1 Prepare the data

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn import datasets
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)   # so that everybody gets similar results

In [ ]:
digits = datasets.load_digits()   # 1797 images of size 8x8, with labels 0-9
print(digits.images.shape)
print(digits.target[:10])

> **Exercise B1**: Plot the first 10 images of `digits.images` in a row (use `plt.subplots(1, 10)`) and write the true label (`digits.target`) above each image.

In [ ]:
# TODO


> **Exercise B2**: Split the images and labels into a training set (70%) and a test set (30%), as in Tutorial 2. Use `random_state=42`.

In [ ]:
X_train, X_test, y_train, y_test = None, None, None, None  # TODO


Before the network can use the images we have to **normalise** them (the pixel values go from 0 to 16 — we transform them to the range 0 to 1), **reshape** them (the CNN expects the layout *(number of images, height, width, number of channels)* — our grey-scale images have 1 channel) and **one-hot encode** the target.

Instead of the class number (e.g. `3`) we then have a vector of length 10: the i-th component is 1 if the image belongs to class i, and all other components are 0. This transforms the categorical variable into a numerical form. This is already done for you — run the cell and check the output.

In [ ]:
X_train = X_train / 16.0
X_test = X_test / 16.0

X_train = X_train.reshape((X_train.shape[0], 8, 8, 1))
X_test = X_test.reshape((X_test.shape[0], 8, 8, 1))

y_train_labels, y_test_labels = y_train, y_test      # keep the original labels for later
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

print("Images:", X_train.shape, X_test.shape)
print("Label of the first training image:", y_train_labels[0], "->", y_train[0])

### 6.2 Build the model

The core data structures of *Keras* are **layers** and **models**. We use the **Sequential** model: a simple stack of layers. We stack layers with the `add()` method.

**Convolutional layer** — the layer's parameters are a set of learnable *filters* (exactly like our edge kernels, but with numbers that are learned!). We slide each filter across the width and height of the input and compute dot products between the filter and the input at every position. This produces an *activation map* that shows where the filter "found" its pattern.

```python
Conv2D(filters=16, kernel_size=(3,3), strides=(1,1), activation='relu')
# if width and height are equal you can write in short
Conv2D(filters=16, kernel_size=3, strides=1, activation='relu')
```

Here, `filters` is the number of filters, `kernel_size` is the height and width of one filter, and `activation` is the function applied afterwards (`relu` sets negative values to 0).

> **Exercise B3**: Add **at least one** convolutional layer to the model below. The last layers are already there: `Flatten` turns the images into one long vector, and the output layer has 10 units (one per digit) with the *softmax* activation, which turns the outputs into class probabilities.

In [ ]:
model = Sequential()
model.add(Input(shape=(8, 8, 1)))

# TODO: add at least 1 convolutional layer

model.add(Flatten())
model.add(Dense(10, activation='softmax'))

model.summary()

> **Interpretation exercise**
>
> Look at the `Param #` column of `model.summary()`. Where does the number of parameters of your *first* convolutional layer come from? (Hint: every filter has `kernel height × kernel width × input channels` numbers, plus one extra number per filter.) Which layer has the most parameters?

*Your answer:*

> ...


### 6.3 Compile the model

To train the network we need to *compile* it, which means choosing a **loss function** and an **optimiser**.

**Loss function** — computes the quantity that the model should seek to minimise during training. It calculates how poorly the model is performing by comparing what it predicts with the value it is supposed to output. The choice depends on the task:
- `BinaryCrossentropy()` — classification with two classes
- `CategoricalCrossentropy()` — classification with several classes (multi-class)
- `MeanSquaredError()` — regression

**Optimiser** — the goal of optimisation is to find the parameters that minimise the loss. For this we compute the direction in which the loss decreases fastest (the gradient) and change the parameters a little bit in that direction.
- `SGD()` — stochastic gradient descent: gradient descent on small batches of examples
- `Adam()` — adaptive moment estimation, which also uses fractions of previous gradients (momentum)

> **Exercise B4**: Compile your model. We have a multi-class problem, so you cannot use the binary cross-entropy loss. Use `metrics=['accuracy']` so that we can follow the accuracy during training.

In [ ]:
# TODO: import a loss function from tensorflow.keras.losses
# TODO: import an optimiser from tensorflow.keras.optimizers

# TODO: compile your model with a loss, an optimiser and metrics=['accuracy']


### 6.4 Train the model

To train the model we use the `fit()` method. We have to provide the *training data* (images and labels), the **number of epochs** (how often the network sees the whole training set) and the **batch size** (how many images are used for one update of the parameters).

Instead of a separate validation set we use `validation_split`, which sets aside the last x% of the training data. After every epoch the model is also evaluated on this validation data — so we can see whether it also works on images it is *not* trained on. The `fit()` method returns a `History` object that records the loss and accuracy of every epoch.

> **Exercise B5**: Train your model with `validation_split=0.1`. Choose the number of epochs and the batch size yourself (a good start: 20 epochs, batch size 32).

In [ ]:
epochs = None      # TODO
batch_size = None  # TODO

history = None     # TODO: model.fit(...)


The `History` object lets us look at the training: the plots show the **loss** (smaller is better) and the **accuracy** (higher is better), for the training data and for the validation data.

In [ ]:
def plot_history(history):
    h = history.history
    epochs_range = range(1, len(h['loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(epochs_range, h['loss'], label='training loss')
    ax1.plot(epochs_range, h['val_loss'], label='validation loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
    ax2.plot(epochs_range, h['accuracy'], label='training accuracy')
    ax2.plot(epochs_range, h['val_accuracy'], label='validation accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_ylim(0, 1.05); ax2.legend()
    plt.tight_layout()
    plt.show()

plot_history(history)

> **Interpretation exercise**
>
> Describe the training and the validation curves. Do they behave the same way? Is the model still improving at the end, or did it stop learning earlier? Recall from the lecture and Tutorial 2 why we always check a model on data it was not trained on — do you see signs of over-fitting here?

*Your answer:*

> ...


### 6.5 Evaluate and predict

Training and validation data were used while building the model. The honest test is the **test set**, which the model has never seen.

> **Exercise B6**: Calculate the loss and the accuracy on the test set with `model.evaluate(...)`.

In [ ]:
# TODO


A single accuracy number hides *which* mistakes the model makes. As in Tutorial 2, a confusion matrix shows this. The model outputs 10 probabilities per image; the predicted digit is the one with the highest probability (`np.argmax`).

> **Exercise B7**: Get the predicted digit for every test image into `y_pred` (hint: `model.predict(X_test)` gives the probabilities, then use `np.argmax(..., axis=1)`).

In [ ]:
y_pred = None  # TODO


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test_labels, y_pred)
plt.show()

> **Exercise B8**: Find the test images that the model got wrong (hint: `np.where(y_pred != y_test_labels)[0]`) and plot up to 8 of them, with the true and the predicted label in the title.

In [ ]:
# TODO


> **Interpretation exercise**
>
> Which digits does the model confuse most often? Could *you* read the wrongly classified images? Why might 8×8 pixels be a problem?

*Your answer:*

> ...


### 6.6 Experiments

Now you can play scientist. The helper below trains a fresh model quickly and reports the accuracy on training, validation and test data. `build_cnn(filters)` creates a small CNN like yours.

In [ ]:
def build_cnn(filters=32):
    return Sequential([
        Input(shape=(8, 8, 1)),
        Conv2D(filters, kernel_size=3, activation='relu'),
        Conv2D(filters * 2, kernel_size=3, activation='relu'),
        Flatten(),
        Dense(10, activation='softmax'),
    ])

def run_experiment(model, X=X_train, y=y_train, epochs=20, batch_size=32):
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    history = model.fit(X, y, validation_split=0.1, epochs=epochs, batch_size=batch_size, verbose=0)
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f"parameters: {model.count_params():>7} | "
          f"train acc: {history.history['accuracy'][-1]:.3f} | "
          f"val acc: {history.history['val_accuracy'][-1]:.3f} | "
          f"test acc: {test_acc:.3f}")
    return history

_ = run_experiment(build_cnn())

> **Exercise B9 — Do we need convolutions at all?** Build a model *without* convolutional layers: `Flatten()`, then one `Dense` layer with 64 units and `relu`, then the output layer. Train it with `run_experiment` and compare parameters and accuracy with the CNN above.

In [ ]:
dense_model = Sequential([
    Input(shape=(8, 8, 1)),
    # TODO
])

_ = run_experiment(dense_model)

> **Exercise B10 — How much data do we need?** Train the CNN again, but only on the **first 100** training images (`X_train[:100]`, `y_train[:100]`). Look at the training accuracy and the test accuracy. What do you see?

In [ ]:
# TODO: run_experiment(build_cnn(), X=..., y=...)


> **Interpretation exercise**
>
> What happened in B10, and how does it relate to the idea of over-fitting from the lecture and Tutorial 2 (good on training data, worse on new data)? Now think back to Part A: the MRI network had to be trained on real brain scans. What does this experiment tell you about how many scans (and what kind) you would need?

*Your answer:*

> ...


### 6.7 Challenge: the full MNIST data set

The digits above are a small version of the famous **MNIST** data set: 70,000 images of handwritten digits with 28×28 pixels. Can you reach a test accuracy **above 98%**?

Training on 28×28 images is much slower on a CPU. Two tips: **(1)** use few epochs (2–3 are enough) — and if available switch Colab to a GPU (*Runtime → Change runtime type → T4 GPU*; then run the notebook from the top again); **(2)** use `MaxPooling2D(pool_size=2)` layers after convolutional layers: they halve the image size, which makes the network much faster.

The data is prepared for you:

In [ ]:
from tensorflow.keras.datasets import mnist

(X_train_big, y_train_big), (X_test_big, y_test_big) = mnist.load_data()   # 60,000 training + 10,000 test images
X_train_big = (X_train_big / 255.0).astype('float32').reshape((-1, 28, 28, 1))
X_test_big = (X_test_big / 255.0).astype('float32').reshape((-1, 28, 28, 1))
y_train_big = to_categorical(y_train_big)
y_test_big = to_categorical(y_test_big)

print(X_train_big.shape, X_test_big.shape)

> **Exercise B11**: Build, compile and train a CNN with input shape (28, 28, 1) on the big data set (use `validation_split=0.1`), and evaluate it on `X_test_big`.

In [ ]:
# TODO


***
## 7. Discussion

In Part A we saw that a photo and an MRI scan are nothing but arrays of numbers, that a *filter* slides over such an array and highlights patterns, and that a CNN stacks many learned filters — which let a pretrained network give a score for a brain scan in a fraction of a second. In Part B we built and trained such a network ourselves and looked at what it gets wrong.

Two lessons connect directly to the rest of the course. First: **a model is only as good as its evaluation** — a training curve or a single prediction on a single scan says very little; what counts is performance on data the model has never seen (Tutorial 2). Second: **a network that produces a number does not explain itself** — before a model like the one in Part A is used with real patients, we have to ask what data it was trained on, how it was validated, and for whom it works (the topic of the ethics session).

## 8. Quiz

1. What is a convolutional filter, and how do the filters in a CNN differ from the edge filters we wrote by hand?
2. Why did we have to mask and resize the MRI before giving it to the network in Part A?
3. What do the training and the validation curve tell you, and what does it mean if they drift apart?
4. Why is one correct prediction on one MRI scan not evidence that a network is a good diagnostic tool?

***